# Bloomberg pull smoke test (`bdib`)

Ruta fija: `fwd_screener_v1`. Antes de correr:
```bash
cd /v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1
git checkout main && git pull origin main
```

In [ ]:
import os, sys, importlib, logging
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

# Mute bbgapi INFO (pink/red Jupyter log blocks)
logging.disable(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
for _n in list(logging.root.manager.loggerDict):
    lg = logging.getLogger(_n)
    if any(k in str(_n).lower() for k in ("bbg", "blp", "dz", "datazone", "activity", "ms")):
        lg.disabled = True
        lg.handlers = []

REPO = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
if not REPO.exists():
    raise FileNotFoundError("Missing clone: %s" % REPO)

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]

rp = REPO / "run_prediction.py"
print("REPO:", REPO)
print("python:", sys.executable)
print("pandas:", pd.__version__)
print("run_prediction:", rp.resolve())
if not rp.exists():
    raise FileNotFoundError("run_prediction.py not in %s — git pull origin main" % REPO)

# Auto-patch old pandas-0.25-incompatible attrs usage if clone is stale
src = rp.read_text()
if "table.attrs" in src:
    src2 = src.replace(
        'table.attrs["asof"] = asof\n    table.attrs["text"] = text\n    return table',
        'LAST_ASOF = asof\n    LAST_TEXT = text\n    return table',
    )
    if "LAST_ASOF" not in src2:
        src2 = "LAST_ASOF = \"\"\nLAST_TEXT = \"\"\n\n" + src2
    if "global LAST_ASOF" not in src2 and "def run_once" in src2:
        src2 = src2.replace(
            "def run_once(*, refresh: bool = True, force_history: bool = False) -> pd.DataFrame:\n",
            "def run_once(*, refresh: bool = True, force_history: bool = False) -> pd.DataFrame:\n"
            "    global LAST_ASOF, LAST_TEXT\n\n",
        )
    # strip remaining attrs usages in email path
    src2 = src2.replace('t.attrs.get("asof", "")', "LAST_ASOF")
    src2 = src2.replace("t.attrs.get('asof', '')", "LAST_ASOF")
    src2 = src2.replace('t.attrs.get("text")', "LAST_TEXT")
    src2 = src2.replace("t.attrs.get('text')", "LAST_TEXT")
    if "table.attrs" in src2:
        raise RuntimeError("Could not auto-patch table.attrs — run: git pull origin main")
    rp.write_text(src2)
    print("PATCHED", rp, "(removed table.attrs)")
else:
    print("run_prediction.py OK (no table.attrs)")

In [ ]:
# 1) Cliente
from data import _quiet_bbg_logs, _bquery
_quiet_bbg_logs()
bquery = _bquery()
print("started", bquery)

In [ ]:
# 2) Smoke bdib (1 ticker)
end = datetime(2026, 9, 8, 21, 0, 0)
start = end - timedelta(days=100)
fmt = "%Y-%m-%dT%H:%M:%S"

bars = bquery.bdib(
    security="USOSFR10 BGN Curncy",
    event_type="TRADE",
    interval=180,
    start_datetime=start.strftime(fmt),
    end_datetime=end.strftime(fmt),
)
print(bars.shape, list(bars.columns))
bars.head()

In [ ]:
# 3) Historia vía data.py
import data
importlib.reload(data)
from data import _securities, pull_history_into_store, append_live_bar, load_live_store
from curve_config import BBG_INTERVAL_MINUTES, LIVE_STORE_PATH

print("n_securities", len(_securities()), "interval", BBG_INTERVAL_MINUTES)
hist = pull_history_into_store(
    start_date=start.strftime("%Y-%m-%d"),
    end_date=end.strftime("%Y-%m-%d %H:%M:%S"),
    freq=180,
)
print(hist.shape, hist.index[0], "→", hist.index[-1])
hist.iloc[:, :6].tail()

In [ ]:
# 4) Append + predicción
panel = append_live_bar()
print("store", LIVE_STORE_PATH, "n", len(load_live_store()), "last", panel.index[-1])

import run_prediction
importlib.reload(run_prediction)
table = run_prediction.run_once(refresh=False)
cols = [c for c in ["model", "structure", "nivel", "y_hat_now", "residual", "y_hat_next"] if c in table.columns]
table[cols].head(12)